In [1]:
import asyncio
import logging
import os
from pathlib import Path
import yaml
import pandas as pd
from typing import Dict, Any, Tuple, List, Optional
from dotenv import load_dotenv
import argparse
from datetime import datetime

from src.factories.clothing_factory import ClothingFactory
from src.services.api_service import APIService
from src.services.attribute_generator import AttributeGenerator
from src.services.batch_processor import BatchProcessor
from src.utils.logging_utils import (
    setup_logging, 
    LoggingContext, 
    PerformanceMetrics,
    StructuredLogger
)
from src.services.product_standardizer import ProductStandardizer
from src.base.clothing_item import ClothingItem


In [2]:
from abc import ABC, abstractmethod
from enum import Enum
from typing import List, Dict, Any, Union
import re
import logging
from pydantic import (
    BaseModel, Field, field_validator, ValidationInfo
)
from src.base.enums import (
    Color, ColorDetailed, Pattern, Material, 
    EmbellishmentLevel, Embellishment, Occasion, 
    Style, Gender, AgeGroup
)
from src.utils.validation import validate_enum_field, validate_enum_list
from src.utils.config_loader import ConfigManager
import pandas as pd


In [3]:
class ConfigLoader:
    def __init__(self, config_dir: Path):
        self.config_dir = config_dir
        
    def load(self) -> Dict[str, Any]:
        """Load base configuration from YAML file"""
        config_path = self.config_dir / "base_config.yaml"
        with open(config_path) as f:
            return yaml.safe_load(f)


def load_config(config_path: str) -> Dict:
    with open(config_path) as f:
        return yaml.safe_load(f)

In [4]:
ClothingFactory.initialize()

## Load products

In [10]:
base_path = '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/product_attributes/config/base_config.yaml'
config = load_config(base_path)
from src.utils.config_loader import ConfigManager

In [11]:
def load_and_prepare_data(config: Dict) -> Tuple[pd.DataFrame, Dict]:
    input_path = config["paths"]["input"]["product_data"]
    
    # Read CSV with product_id as string
    df = pd.read_csv(
        input_path,
        dtype={
            "Product ID": str,  # Force product_id to be string
            "Brand Name": str,
            "Product Type": str,
            "Product Name": str,
            "Product Description": str,
            "Product Image Link": str,
            "Price": float,
            "Size": str,
            "Tags": str,
            "Product URL": str
        }
    )
    
    # Define column mappings
    column_mappings = {
        "Brand Name": "brand_name",
        "Product ID": "product_id", 
        "Product Type": "product_type",
        "Product Name": "title",
        "Product Description": "description",
        "Product Image Link": "product_image_link",
        "Price": "price",
        "Size": "size",
        "Tags": "tags",
        "Product URL": "product_url"
    }
    
    # Rename columns
    df = df.rename(columns=column_mappings)
    
    # Ensure product_id is string type
    df['product_id'] = df['product_id'].astype(str)
    
    # Remove any .0 suffix from product IDs if they were originally floats
    df['product_id'] = df['product_id'].apply(
        lambda x: x.rstrip('.0') if x.endswith('.0') else x
    )
    
    # Standardize product types for all products
    df['standardized_category'] = df['product_type'].apply(
        lambda x: ProductStandardizer.standardize_product(
            {'product_type': x}
        )['standardized_category']
    )
    
    # Get image paths
    image_paths = {}
    images_dir = Path(config["paths"]["input"].get("images_dir")) 
    if 'brand' in config.get('attributes', {}):
        images_dir = images_dir / config['attributes']['brand']['default']
    
    if images_dir and images_dir.exists():
        for product_id in df["product_id"]:
            potential_paths = [
                images_dir / f"{product_id}.jpg",
                images_dir / f"{product_id}.png",
                images_dir / f"{product_id}.webp",
                images_dir / product_id / "main.jpg",
                images_dir / product_id / "main.png"
            ]
            
            for path in potential_paths:
                if path.exists():
                    image_paths[product_id] = str(path)
                    break
    
    return df, image_paths


def filter_products(
    df: pd.DataFrame,
    available_types: List[str],
    target_types: Optional[List[str]] = None
) -> pd.DataFrame:
    """
    Filter products based on standardized product categories.
    """
    available_types = [t.title() for t in available_types]

    # Standardize all product types
    df['standardized_category'] = df['product_type'].apply(
        lambda x: ProductStandardizer.standardize_product({'product_type': x})['standardized_category']
    )
    
    # If target types specified, filter for those first
    if target_types:
        target_standardized = [
            ProductStandardizer.standardize_product({'product_type': t})['standardized_category']
            for t in target_types
        ]
        target_standardized = list(set(target_standardized))
        filtered_df = df[df['standardized_category'].isin(target_standardized)]
    else:
        filtered_df = df.copy()

    # Split into standard and non-standard products
    standard_products = filtered_df[filtered_df['standardized_category'].isin(available_types)]
    other_products = filtered_df[~filtered_df['standardized_category'].isin(available_types)]
    
    if not other_products.empty:
        print(
            f"Found {len(other_products)} products with non-standard categories. "
            "Will use base configuration for these."
        )
        
    # Combine both dataframes, maintaining the separation via standardized_category
    filtered_df = pd.concat([standard_products, other_products])
    
    # Log distribution
    if not filtered_df.empty:
        type_counts = filtered_df['standardized_category'].value_counts().to_dict()
        print(
            f"Category distribution {type_counts}"
        )
    
    return filtered_df

In [25]:
input_path = config["paths"]["input"]["product_data"]

# Read CSV with product_id as string
df = pd.read_csv(
    input_path,
    dtype={
        "Product ID": str,  # Force product_id to be string
        "Brand Name": str,
        "Product Type": str,
        "Product Name": str,
        "Product Description": str,
        "Product Image Link": str,
        "Price": float,
        "Size": str,
        "Tags": str,
        "Product URL": str
    }
)

# Define column mappings
column_mappings = {
    "Brand Name": "brand_name",
    "Product ID": "product_id", 
    "Product Type": "product_type",
    "Product Name": "title",
    "Product Description": "description",
    "Product Image Link": "product_image_link",
    "Price": "price",
    "Size": "size",
    "Tags": "tags",
    "Product URL": "product_url"
}

# Rename columns
df = df.rename(columns=column_mappings)

# Ensure product_id is string type
df['product_id'] = df['product_id'].astype(str)

# Remove any .0 suffix from product IDs if they were originally floats
df['product_id'] = df['product_id'].apply(
    lambda x: x.rstrip('.0') if x.endswith('.0') else x
)

# Standardize product types for all products
df['standardized_category'] = df['product_type'].apply(
    lambda x: ProductStandardizer.standardize_product(
        {'product_type': x}
    )['standardized_category']
)

# Get image paths
# Initialize empty dict for image paths
image_paths = {}

# Get images directory path
images_dir = Path(config["paths"]["input"].get("images_dir"))
if 'brand' in config.get('attributes', {}):
    images_dir = images_dir / config['attributes']['brand']['default']

# Debug prints
print(f"Images directory path: {images_dir}")
print(f"Images directory exists: {images_dir.exists()}")

if images_dir and images_dir.exists():
    print("Directory exists")
    print(f"Number of product IDs to process: {len(df['product_id'])}")
    
    for product_id in df["product_id"]:
        potential_paths = [
            images_dir / f"{product_id}.jpg",
            images_dir / f"{product_id}.png", 
            images_dir / f"{product_id}.webp",
            images_dir / product_id / "main.jpg",
            images_dir / product_id / "main.png"
        ]
        
        # Debug print paths being checked
        print(f"\nChecking paths for product {product_id}:")
        for path in potential_paths:
            print(f"Checking {path}, exists: {path.exists()}")
            if path.exists():
                image_paths[product_id] = str(path)
                print(f"Found image at {path}")
                break

print(f"\nTotal images found: {len(image_paths)}")

Images directory path: /Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images/suta
Images directory exists: True
Directory exists
Number of product IDs to process: 4888

Checking paths for product 7355069300801:
Checking /Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images/suta/7355069300801.jpg, exists: True
Found image at /Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images/suta/7355069300801.jpg

Checking paths for product 7372922781761:
Checking /Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images/suta/7372922781761.jpg, exists: True
Found image at /Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images/suta/7372922781761.jpg

Checking paths for product 737287654

In [27]:
image_paths

{'7355069300801': '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images/suta/7355069300801.jpg',
 '7372922781761': '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images/suta/7372922781761.jpg',
 '7372876546113': '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images/suta/7372876546113.jpg',
 '7372801835073': '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images/suta/7372801835073.jpg',
 '7372794265665': '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images/suta/7372794265665.jpg',
 '7372774015041': '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images/suta/7372774015041.jpg',
 '7372733612097': '/Us

In [21]:
images_dir

PosixPath('/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images/suta')

In [29]:
target_types = ['jacket']
available_types = ['dress', 'saree', 'kurta', 'dupatta', 'trouser', 'blouse', 'shirt', 'lehenga']

In [30]:
available_types = [t.title() for t in available_types]
# Standardize all product types
df['standardized_category'] = df['product_type'].apply(
    lambda x: ProductStandardizer.standardize_product({'product_type': x})['standardized_category']
)

# If target types specified, filter for those first
if target_types:
    target_standardized = [
        ProductStandardizer.standardize_product({'product_type': t})['standardized_category']
        for t in target_types
    ]
    target_standardized = list(set(target_standardized))
    filtered_df = df[df['standardized_category'].isin(target_standardized)]
else:
    filtered_df = df.copy()

# # Split into standard and non-standard products
standard_products = filtered_df[filtered_df['standardized_category'].isin(available_types)]
other_products = filtered_df[~filtered_df['standardized_category'].isin(available_types)]

if not other_products.empty:
    print(
        f"Found {len(other_products)} products with non-standard categories. "
        "Will use base configuration for these."
    )
    
# Combine both dataframes, maintaining the separation via standardized_category
filtered_df = pd.concat([standard_products, other_products])

# Log distribution
if not filtered_df.empty:
    type_counts = filtered_df['standardized_category'].value_counts().to_dict()
    print(
        f"Category distribution {type_counts}"
    )


Found 11 products with non-standard categories. Will use base configuration for these.
Category distribution {'Jacket': 11}


In [27]:
other_products

,brand_name,product_id,product_type,title,description,product_image_link,price,size,tags,product_url,standardized_category


In [8]:
from src.base.clothing_item import ClothingItem 
# patterns, default_strategy = ClothingItem.get_category_mapping()

In [9]:
config = ConfigManager.get_base_config()
standardization_config = config.get('category_standardization', {})

patterns = []
for item in standardization_config.get('patterns', []):
    try:
        combined_pattern = r'\b(?:{})\b'.format('|'.join(item['matches']))
        compiled_pattern = re.compile(combined_pattern, re.IGNORECASE)
        patterns.append((compiled_pattern, item['name'].title()))
    except re.error as e:
        print(f"Invalid regex pattern for {item['name']}: {e}")
        continue



In [22]:
df

,brand_name,product_id,product_type,title,description,product_image_link,price,size,tags,product_url,standardized_category
0,suta,7355069300801,Rakhi,Sneh ka dhaaga,Details Dimension of Rakhi : Length - 0.45 m (...,['https://cdn.shopify.com/s/files/1/0026/6544/...,350.0,['Default Title'],"Accessories, BLUE, Colour_Blue, discount_appli...",https://suta.in/products/sneh-ka-dhaaga-rakhi,Rakhi
1,suta,7372922781761,garage_saree,GAR-1967,DetailsLength: 5.50 m (550.00 cm) ; Width: 1....,['https://cdn.shopify.com/s/files/1/0026/6544/...,2765.0,['Default Title'],"Colour_Multicolour, Fabric_Mul Cotton, lgsy_no...",https://suta.in/products/gar-1967,Saree
2,suta,7372876546113,garage_saree,GAR-1960,DetailsLength: 5.50 m (550.00 cm) ; Width: 1....,['https://cdn.shopify.com/s/files/1/0026/6544/...,2520.0,['Default Title'],"BLUE, Blue Saree, Blue Sari, Blue Saris, Colou...",https://suta.in/products/gar-1960,Saree
3,suta,7372801835073,garage_saree,GAR-1955,DetailsLength: 5.30 m (530.00 cm) ; Width: 1....,['https://cdn.shopify.com/s/files/1/0026/6544/...,16940.0,['Default Title'],"Colour_White, Fabric_Organza, lgsy_non_exchang...",https://suta.in/products/gar-1955,Saree
4,suta,7372794265665,garage_saree,GAR-1954,DetailsLength: 5.45 m (545.00 cm) ; Width: ...,['https://cdn.shopify.com/s/files/1/0026/6544/...,3255.0,['Default Title'],"Colour_Pink, Fabric_Modal Viscose, lgsy_non_ex...",https://suta.in/products/gar-1954,Saree
...,...,...,...,...,...,...,...,...,...,...,...
4883,suta,7385414500417,Saree,Kumkum Laali,DetailsLength: 5.50 m (550 cm) ; Width: 1.143 ...,['https://cdn.shopify.com/s/files/1/0026/6544/...,3800.0,['Default Title'],"__with:pre-drape-this-saree, Agomoni, Agomoni ...",https://suta.in/products/kumkum-laali-saree,Saree
4884,suta,7385412829249,Saree,Laal Joba,DetailsLength: 5.50 m (550 cm) ; Width: 1.143 ...,['https://cdn.shopify.com/s/files/1/0026/6544/...,9200.0,['Default Title'],"__with:pre-drape-this-saree, Agomoni, Agomoni ...",https://suta.in/products/laal-joba-saree,Saree
4885,suta,7385411059777,Saree,Pujo Pujo Gondho,DetailsLength: 5.50 m (550 cm) ; Width: 1.143 ...,['https://cdn.shopify.com/s/files/1/0026/6544/...,5200.0,['Default Title'],"__with:pre-drape-this-saree, Agomoni, Agomoni ...",https://suta.in/products/pujo-pujo-gondho-saree,Saree
4886,suta,7385406832705,Saree,Roktima,DetailsLength: 5.50 m (550 cm) ; Width: 1.143 ...,['https://cdn.shopify.com/s/files/1/0026/6544/...,3450.0,['Default Title'],"__with:pre-drape-this-saree, Agomoni, Agomoni ...",https://suta.in/products/roktima-saree,Saree


In [21]:
temp = filter_products(df,available_types = ['dress', 'saree', 'kurta', 'dupatta', 'trouser', 'blouse', 'shirt', 'lehenga']) 
temp

Found 4888 products with non-standard categories. Will use base configuration for these.
Category distribution {'Saree': 2670, 'Blouse': 1221, 'Kurta': 155, 'Dress': 154, 'Accessories': 148, 'Fabric': 85, 'Underskirt': 63, 'Combo Sets': 60, 'Shirt': 59, '': 48, 'Home Textiles': 35, 'Skirt': 32, 'Dupatta': 31, 'Trouser': 20, 'Co-Ord Set': 18, 'Rakhi': 14, 'Painting': 13, 'Mittens': 11, 'Lehenga': 11, 'Jacket': 11, 'Handkerchief/Table Napkin': 8, 'Gift Items': 7, 'Mask Chain': 4, 'Bag': 4, 'Gamcha': 2, 'Shorts': 1, 'Underlay Top': 1, 'Underlay Pant': 1, 'Handkerchief': 1}


,brand_name,product_id,product_type,title,description,product_image_link,price,size,tags,product_url,standardized_category
0,suta,7355069300801,Rakhi,Sneh ka dhaaga,Details Dimension of Rakhi : Length - 0.45 m (...,['https://cdn.shopify.com/s/files/1/0026/6544/...,350.0,['Default Title'],"Accessories, BLUE, Colour_Blue, discount_appli...",https://suta.in/products/sneh-ka-dhaaga-rakhi,Rakhi
1,suta,7372922781761,garage_saree,GAR-1967,DetailsLength: 5.50 m (550.00 cm) ; Width: 1....,['https://cdn.shopify.com/s/files/1/0026/6544/...,2765.0,['Default Title'],"Colour_Multicolour, Fabric_Mul Cotton, lgsy_no...",https://suta.in/products/gar-1967,Saree
2,suta,7372876546113,garage_saree,GAR-1960,DetailsLength: 5.50 m (550.00 cm) ; Width: 1....,['https://cdn.shopify.com/s/files/1/0026/6544/...,2520.0,['Default Title'],"BLUE, Blue Saree, Blue Sari, Blue Saris, Colou...",https://suta.in/products/gar-1960,Saree
3,suta,7372801835073,garage_saree,GAR-1955,DetailsLength: 5.30 m (530.00 cm) ; Width: 1....,['https://cdn.shopify.com/s/files/1/0026/6544/...,16940.0,['Default Title'],"Colour_White, Fabric_Organza, lgsy_non_exchang...",https://suta.in/products/gar-1955,Saree
4,suta,7372794265665,garage_saree,GAR-1954,DetailsLength: 5.45 m (545.00 cm) ; Width: ...,['https://cdn.shopify.com/s/files/1/0026/6544/...,3255.0,['Default Title'],"Colour_Pink, Fabric_Modal Viscose, lgsy_non_ex...",https://suta.in/products/gar-1954,Saree
...,...,...,...,...,...,...,...,...,...,...,...
4883,suta,7385414500417,Saree,Kumkum Laali,DetailsLength: 5.50 m (550 cm) ; Width: 1.143 ...,['https://cdn.shopify.com/s/files/1/0026/6544/...,3800.0,['Default Title'],"__with:pre-drape-this-saree, Agomoni, Agomoni ...",https://suta.in/products/kumkum-laali-saree,Saree
4884,suta,7385412829249,Saree,Laal Joba,DetailsLength: 5.50 m (550 cm) ; Width: 1.143 ...,['https://cdn.shopify.com/s/files/1/0026/6544/...,9200.0,['Default Title'],"__with:pre-drape-this-saree, Agomoni, Agomoni ...",https://suta.in/products/laal-joba-saree,Saree
4885,suta,7385411059777,Saree,Pujo Pujo Gondho,DetailsLength: 5.50 m (550 cm) ; Width: 1.143 ...,['https://cdn.shopify.com/s/files/1/0026/6544/...,5200.0,['Default Title'],"__with:pre-drape-this-saree, Agomoni, Agomoni ...",https://suta.in/products/pujo-pujo-gondho-saree,Saree
4886,suta,7385406832705,Saree,Roktima,DetailsLength: 5.50 m (550 cm) ; Width: 1.143 ...,['https://cdn.shopify.com/s/files/1/0026/6544/...,3450.0,['Default Title'],"__with:pre-drape-this-saree, Agomoni, Agomoni ...",https://suta.in/products/roktima-saree,Saree


In [ ]:
filter

## Count products

In [8]:
#!/usr/bin/env python3

import json
import os
from typing import Dict, Any, Optional


def float_hook(dct: dict) -> dict:
    for k, v in dct.items():
        if isinstance(v, str):
            try:
                if 'e' in v.lower():
                    dct[k] = float(v)
            except ValueError:
                pass
        elif isinstance(v, list):
            for i, item in enumerate(v):
                if isinstance(item, str) and 'e' in item.lower():
                    try:
                        v[i] = float(item)
                    except ValueError:
                        pass
    return dct


def count_unique_products(file_path: str) -> int:
    try:
        all_data = {}
        current_obj = ""
        brace_count = 0
        total_objects_found = 0
        failed_objects = 0
        
        with open(file_path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                    
                brace_count += line.count('{') - line.count('}')
                current_obj += line
                
                if brace_count == 0 and current_obj:
                    total_objects_found += 1
                    try:
                        data = json.loads(current_obj, object_hook=float_hook)
                        if data:
                            product_id = next(iter(data))
                            all_data[product_id] = data[product_id]
                    except json.JSONDecodeError:
                        failed_objects += 1
                        preview_len = 75
                        truncated = len(current_obj) > preview_len
                        preview = current_obj[:preview_len]
                        if truncated:
                            preview += "..."
                        obj_num = total_objects_found
                        print(f"\nFailed to parse object {obj_num}:")
                        print(preview)
                    current_obj = ""
        
        product_count = len(all_data)
        print("\nParsing Statistics:")
        print(f"Total JSON objects found: {total_objects_found}")
        print(f"Failed to parse: {failed_objects}")
        print(f"Unique products found: {product_count}")
        
        if product_count > 0:
            print("\nFirst few product IDs found:")
            for pid in list(all_data.keys())[:5]:
                print(f"- {pid}")
                
        return product_count
            
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return 0
    except Exception as e:
        print(f"Error: {str(e)}")
        return 0


def load_product_by_id(file_path: str, product_id: str) -> Optional[Dict[str, Any]]:
    try:
        current_obj = ""
        brace_count = 0
        
        with open(file_path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                    
                brace_count += line.count('{') - line.count('}')
                current_obj += line
                
                if brace_count == 0 and current_obj:
                    try:
                        data = json.loads(current_obj, object_hook=float_hook)
                        if data and product_id in data:
                            return data[product_id]
                    except json.JSONDecodeError:
                        pass
                    current_obj = ""
        
        print(f"Product ID {product_id} not found")
        return None
            
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return None
    except Exception as e:
        print(f"Error: {str(e)}")
        return None


json_path = 'D:/OneDrive - StarHub Ltd/5. Scripts/product-attributes/output/Saree_attributes.json'
count = count_unique_products(json_path)
print(f"\nFinal count of unique products: {count}")


Parsing Statistics:
Total JSON objects found: 25
Failed to parse: 0
Unique products found: 25

First few product IDs found:
- 7372922781761
- 7372876546113
- 7372801835073
- 7372794265665
- 7372774015041

Final count of unique products: 25


In [9]:
product_temp = load_product_by_id(json_path, '7356361736257')
if product_temp:
    product_temp.pop('text_embedding', None)
    product_temp.pop('image_embedding', None)
product_temp

{'primary_color': 'Red',
 'primary_color_detailed': 'Crimson',
 'secondary_colors': [],
 'secondary_colors_detailed': [],
 'color_pairings_hex': ['#FF0033', '#FFD700', '#FFC0CB', '#800000', '#FF69B4'],
 'pattern': [],
 'saree_type': 'Banarasi',
 'border_width': 'Wide',
 'border_design': 'Zari',
 'border_design_details': ['Intricate zari work on border',
  'Traditional motifs in gold',
  'Scalloped edge design',
  'Contrast red and gold pattern',
  'Rich metallic sheen'],
 'pallu_design': 'Elaborate',
 'pre_draped': False,
 'material': 'Silk',
 'length': 6.3,
 'width': 1.1,
 'weight': 800,
 'care_instructions': 'Dry clean only. Store in a cool, dry place. Avoid direct sunlight.',
 'occasions': [],
 'embellishment_level': 'Medium',
 'embellishment': [],
 'embellishment_detailed': ['Gold zari woven polka dot pattern',
  'Intricate zari border work',
  'Metallic thread embroidery on pallu',
  'Subtle shimmering effect throughout'],
 'style': [],
 'brand_title': 'Red Dream',
 'title': 'Crim

In [5]:
import asyncio
import logging
import os
from pathlib import Path
import yaml
import pandas as pd
from typing import Dict, Any, Tuple, List, Optional
from dotenv import load_dotenv
import argparse
from datetime import datetime

from src.factories.clothing_factory import ClothingFactory
from src.services.api_service import APIService
from src.services.attribute_generator import AttributeGenerator
from src.services.batch_processor import BatchProcessor
from src.utils.logging_utils import (
    setup_logging, 
    LoggingContext, 
    PerformanceMetrics,
    StructuredLogger
)
from src.services.product_standardizer import ProductStandardizer
from src.base.clothing_item import ClothingItem


In [6]:

async def process_product(
    product_data: Dict[str, Any],
    product_type: str,
    attribute_generator: AttributeGenerator
) -> Optional[Dict[str, Any]]:
    """Process a single product."""
    product_id = product_data.get("product_id", "unknown")
    
    # Create product-specific metrics
    product_metrics = PerformanceMetrics()
    product_metrics.checkpoint("start")
    
    # Create structured logger for this product
    product_logger = StructuredLogger(
        "product_processor",
        extra_fields={
            "product_id": product_id,
            "product_type": product_type
        }
    )
    
    product_logger.info(f"Processing product {product_id}")
    
    # Generate attributes using the API
    try:
        product_metrics.checkpoint("api_call_start")
        generated_attributes = await attribute_generator.generate_attributes(
            product_data=product_data,
            product_type=product_type
        )
        product_metrics.checkpoint("api_call_end")
        
        # Calculate API call duration
        api_duration = product_metrics.measure(
            "api_call", 
            "api_call_start", 
            "api_call_end"
        )
        
        if generated_attributes:
            product_logger.info(
                f"Successfully generated attributes for product {product_id}",
                extra={
                    "duration": api_duration,
                    "attribute_count": len(generated_attributes)
                }
            )
            return generated_attributes
        else:
            product_logger.error(
                f"Failed to generate attributes for product {product_id}",
                extra={"duration": api_duration}
            )
            return None
    except Exception as e:
        product_logger.error(
            f"Error processing product {product_id}: {str(e)}",
            extra={"error_type": type(e).__name__},
            exc_info=True
        )
        return None
    finally:
        # Log total processing time
        total_duration = product_metrics.measure("total")
        product_logger.info(
            "Product processing completed",
            extra={"total_duration": total_duration}
        )



In [ ]:
from src.utils.config_loader import ConfigManager
cf = ConfigManager
temp = cf.get_available_product_types()
temp
# product_config = ConfigManager.get_product_config('saree')
# defined_attributes = product_config.get('attributes', {})
# defined_attributes

In [4]:
ConfigManager.get_product_config('saree')['attributes']['brand']['default']

{'required': True,
 'data_type': 'string',
 'item_type': 'string',
 'default': 'suta',
 'in_search_context': True}

In [5]:
from src.base.clothing_item import ClothingItem
ClothingItem.build_search_context(product_data)

NameError: name 'product_data' is not defined

In [3]:
def test_prompt_loading():
    from pathlib import Path
    from src.services.attribute_generator import AttributeGenerator
    
    # Expanded config with required Cohere settings
    config = {
        "prompts": {
            "dir": "D:/OneDrive - StarHub Ltd/5. Scripts/product-attributes/config/prompts"  # Your actual path
        },
        "api": {
            "cohere": {
                "model": "embed-english-v3.0",  # or whatever model you're using
                "supported_operations": ["embed"]  # Add embed operation
            }
        }
    }
    
    # Create instance with minimal dependencies
    generator = AttributeGenerator(api_service=None, config=config)
    
    # Test prompt loading
    prompt = generator._load_prompt('saree')
    print(prompt)
    return prompt

# Run the test
prompt = test_prompt_loading()

You are a fashion expert specializing in Indian ethnic wear, particularly sarees. 
Analyze this saree image and product information to generate detailed JSON data.

IMPORTANT GUIDELINES:
- Focus ONLY on the saree in the image, not any blouse or other garments shown
- Use the exact product name from the data for brand_title (e.g. if product name is "Green Dream", use that)
- Analyze colors ONLY from the saree itself, not from any accompanying blouse or accessories
- Be precise in identifying embellishments and patterns that are actually present on the saree
- STRICTLY use only the allowed values provided for each attribute
- Use material information from the product data, don't try to infer it from the image
- For ruffled designs, use "Others" as pattern and "Sequinned" as embellishment
- Color pairings MUST be 6-digit hex codes (e.g., "#FF0000", "#00FF00")
- Use the exact product URL from the data for product_url field

Requirements:
1. Colors & Patterns:
   - "primary_color": Identify

In [13]:
from src.factories.clothing_factory import ClothingFactory
from src.base.utils import EnumRegistry

In [14]:
ClothingFactory.initialize()
raw_available_types = ClothingFactory.get_available_types()

In [3]:
from src.base.utils import EnumRegistry
from src.utils.config_loader import ConfigManager

# Debug config loading
config_manager = ConfigManager.get_instance()
print("Base config:", config_manager.get_base_config())
print("Available products:", config_manager.get_available_product_types())
print("Saree config:", config_manager.get_product_config('saree'))

# Get and inspect registry
registry = EnumRegistry.get_instance()
print("\nRegistry state:")
print("Product types:", registry._product_types)
print("Base mappings:", registry._base_mappings)
print("Enum mappings:", registry._enum_mappings)

# Get saree mappings
mappings = registry.get_enum_mappings('kurta')
print("\nkurta mappings:", mappings)

Base config: {'api': {'anthropic': {'model': 'claude-3-5-sonnet-20240620', 'max_retries': 3, 'base_delay': 4, 'max_delay': 60, 'concurrent_limit': 3, 'batch_size': 5, 'pricing': {'base_input': 3.0, 'cache_write': 3.75, 'cache_read': 1.5, 'output': 15.0}}, 'cohere': {'model': 'embed-english-v3.0', 'max_retries': 3, 'base_delay': 4, 'max_delay': 60}}, 'paths': {'input': {'product_data': '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/processed_dataframe.csv', 'images_dir': '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/uplyft-shopify-scraper/data/02_intermediate/suta/images'}, 'output': {'base_dir': '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/product_attributes/output', 'structure': {'attributes': '{product_type}/attributes', 'images': '{product_type}/images'}, 'logs_dir': 'logs'}}, 'prompts': {'dir': 'config/prompts', 'system_temperature': 0.1, 'max_tokens': 4096}, 'attributes': {'brand': {'

In [28]:
c = ClothingFactory()
c.create_product('Jacket')

Using base ClothingItem for undefined type: Jacket
No configuration found for product type: jacket
Error creating Jacket: WOMEN


In [29]:
 available_types = [
                    ClothingItem.standardize_category(t) 
                    for t in ClothingFactory.get_available_types()
                ]

In [33]:
len(df['standardized_category'].unique())

29